# Deep Learning 2025 — Homework Week 8
## RNNs in Practice: Unrolling, Gradient Flow (BPTT), and RNN vs GRU vs LSTM

Please fill the `TODO` blocks.

 ## Overview
 This homework connects your pen-and-paper understanding of RNNs (unrolling, forward pass, BPTT, gates, vanishing gradients)
 to practical experiments in code.

 You will complete **three exercises**:
 1) Implement a **vanilla RNN forward pass** (no training) using NumPy.
 2) Visualize **gradient flow through time** (vanishing/exploding gradients) using PyTorch autograd.
 3) Train **RNN vs GRU vs LSTM** on the same toy long-dependency task and compare behavior.


## Setup

In [ ]:
import math
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Exercise 1 — Vanilla RNN Forward Pass (NumPy)

 We implement the unrolled RNN forward pass:

 $
 h_t = \tanh(W_h h_{t-1} + W_x x_t + b)
 $

 **Goal:** connect the unrolled computation on paper to a working implementation.

 You are given small matrices and a short input sequence.

 ### Tasks
 - Implement `rnn_forward_numpy`.
 - Return all hidden states $(h_1,\ldots,h_T$) as an array of shape (T, hidden_dim).
 - Verify by printing the hidden states.

In [ ]:
# Given small test instance
T = 5
input_dim = 3
hidden_dim = 2

# Fixed weights (chosen arbitrarily for a deterministic test)
W_x = np.array([[ 0.6, -0.2,  0.1],
                [ 0.0,  0.5, -0.4]])  # (hidden_dim, input_dim)
W_h = np.array([[ 0.7,  0.1],
                [-0.3,  0.8]])        # (hidden_dim, hidden_dim)
b   = np.array([0.05, -0.02])         # (hidden_dim,)

# Input sequence x_1..x_T
X_seq = np.array([
    [ 1.0,  0.0,  0.0],
    [ 0.0,  1.0,  0.0],
    [ 0.0,  0.0,  1.0],
    [ 1.0,  1.0,  0.0],
    [ 0.0,  1.0,  1.0],
], dtype=float)  # (T, input_dim)

h0 = np.zeros(hidden_dim)

In [ ]:
# TODO: implement forward pass

def rnn_forward_numpy(X_seq, W_x, W_h, b, h0=None):
    """Compute hidden states for a vanilla RNN.

    Args:
        X_seq: (T, input_dim)
        W_x: (hidden_dim, input_dim)
        W_h: (hidden_dim, hidden_dim)
        b:   (hidden_dim,)
        h0:  (hidden_dim,) initial hidden state (default zeros)

    Returns:
        H: (T, hidden_dim) where H[t] = h_{t+1}
    """
    # TODO
    raise NotImplementedError

H = rnn_forward_numpy(X_seq, W_x, W_h, b, h0=h0)
print("Hidden states (T x hidden_dim):\n", H)

**Reflection:**
 - How does `h_t` depend on *both* the new input and the previous hidden state?
 - Why is the forward pass called *autoregressive* in time?

# Exercise 2 — Gradient Flow Through Time (PyTorch)

 In this exercise, you will *measure* vanishing/exploding gradients through time.

 We'll build a toy task:
 - Input: a sequence of length L with random values.
 - Target: predict the **first element** of the sequence at the final time step.

 This forces the model to preserve information across many steps.

 ### What you will do
 - Build a simple RNN model in PyTorch.
 - Run a single forward+backward pass.
 - Record gradient norms *per time step* for the hidden states.
 - Plot gradient norm vs time step for different sequence lengths.

In [ ]:
# Data generator for the "remember first value" task

def make_memory_task(batch_size=64, seq_len=50):
    """Create sequences x of shape (B, L, 1).
    Target y is the first value x[:,0,0] (shape (B,1)).
    """
    x = torch.randn(batch_size, seq_len, 1)
    y = x[:, 0, :].clone()  # remember the first element
    return x, y

In [ ]:
class SimpleRNNRegressor(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.rnn = nn.RNN(input_size=1, hidden_size=hidden_size, nonlinearity='tanh', batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (B,L,1)
        # We will ALSO return all hidden states for gradient inspection.
        out, hT = self.rnn(x)      # out: (B,L,H)
        y_hat = self.fc(out[:, -1, :])  # use last hidden state
        return y_hat, out

In [ ]:
# TODO: implement gradient-flow measurement

def grad_flow_through_time(seq_len, hidden_size=32, batch_size=64):
    """Runs one forward+backward pass and returns gradient norms per timestep.

    Returns:
        grad_norms: numpy array of shape (seq_len,) where entry t is ||dLoss/dh_t|| averaged over batch.
    """
    model = SimpleRNNRegressor(hidden_size=hidden_size).to(DEVICE)
    criterion = nn.MSELoss()

    x, y = make_memory_task(batch_size=batch_size, seq_len=seq_len)
    x, y = x.to(DEVICE), y.to(DEVICE)

    # Forward
    y_hat, h_all = model(x)  # h_all: (B,L,H)

    # IMPORTANT: we want gradients w.r.t. hidden states
    # TODO: ensure h_all retains gradients

    loss = criterion(y_hat, y)

    # Backward
    model.zero_grad()
    loss.backward()

    # TODO: extract gradients for each time step from h_all.grad
    # Hint: h_all.grad has shape (B,L,H). Take norm over H, then mean over B.

    raise NotImplementedError

In [ ]:
# Run for multiple sequence lengths
seq_lens = [10, 30, 60, 100]
all_grads = {}

for L in seq_lens:
    grads = grad_flow_through_time(seq_len=L, hidden_size=32, batch_size=64)
    all_grads[L] = grads

# Plot
plt.figure()
for L in seq_lens:
    plt.plot(np.arange(1, L+1), all_grads[L], label=f"L={L}")
plt.yscale('log')
plt.xlabel("Time step t")
plt.ylabel(r"$||\partial \mathcal{L} / \partial h_t||$ (log scale)")
plt.title("Gradient flow through time (vanilla RNN)")
plt.legend()
plt.show()

**Reflections:**
1) Do gradients tend to get smaller (vanish) as you go further back in time?  
 2) How does increasing sequence length change the gradient curve?  
 3) Connect what you observe to the pen-and-paper BPTT derivation.

# Exercise 3 — RNN vs GRU vs LSTM on a Long-Dependency Task

 We will train three models on the same memory task:
 - Vanilla RNN
 - GRU
 - LSTM

 Compare training speed and final performance, and relate to gates / vanishing gradients.

 ### Task design
 - Input: sequence (B,L,1)
 - Target: first element (B,1)
 - We train on sequence length L=80 (long enough to be challenging)

 ### What you must do
 - Implement `SequenceRegressor` supporting RNN/GRU/LSTM.
 - Implement a training loop that trains each model and returns loss curves.
 - Plot all loss curves together.

In [ ]:
class SequenceRegressor(nn.Module):
    def __init__(self, cell_type="rnn", hidden_size=64):
        super().__init__()
        self.cell_type = cell_type.lower()
        self.hidden_size = hidden_size

        if self.cell_type == "rnn":
            self.core = nn.RNN(input_size=1, hidden_size=hidden_size, nonlinearity='tanh', batch_first=True)
        elif self.cell_type == "gru":
            self.core = nn.GRU(input_size=1, hidden_size=hidden_size, batch_first=True)
        elif self.cell_type == "lstm":
            self.core = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        else:
            raise ValueError("cell_type must be one of: 'rnn', 'gru', 'lstm'")

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # TODO: forward pass for all three cell types
        # Hint: core(x) returns (out, h) for RNN/GRU; for LSTM it returns (out, (h,c))
        raise NotImplementedError

In [ ]:
def get_dataloader(num_batches=200, batch_size=64, seq_len=80):
    X_list, Y_list = [], []
    for _ in range(num_batches):
        x, y = make_memory_task(batch_size=batch_size, seq_len=seq_len)
        X_list.append(x)
        Y_list.append(y)
    X = torch.cat(X_list, dim=0)
    Y = torch.cat(Y_list, dim=0)
    ds = TensorDataset(X, Y)
    return DataLoader(ds, batch_size=batch_size, shuffle=True)

In [ ]:
# TODO: implement training loop

def train_model(cell_type, seq_len=80, hidden_size=64, epochs=10, lr=1e-3):
    model = SequenceRegressor(cell_type=cell_type, hidden_size=hidden_size).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    loader = get_dataloader(num_batches=200, batch_size=64, seq_len=seq_len)

    losses = []
    t0 = time.time()

    for ep in range(epochs):
        ep_loss = 0.0
        n_batches = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            # TODO: forward, loss, backward, step
            # opt.zero_grad(); y_hat = model(xb); loss = ...; loss.backward(); opt.step()

            n_batches += 1

        losses.append(ep_loss / max(1, n_batches))
        print(f"{cell_type.upper()} | epoch {ep+1}/{epochs} | loss {losses[-1]:.4f}")

    t1 = time.time()
    print(f"{cell_type.upper()} training time: {t1-t0:.2f}s")
    return losses

In [ ]:
# Train and compare
cell_types = ["rnn", "gru", "lstm"]
curves = {}

for ct in cell_types:
    curves[ct] = train_model(ct, seq_len=80, hidden_size=64, epochs=10, lr=1e-3)

plt.figure()
for ct in cell_types:
    plt.plot(curves[ct], label=ct.upper())
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("RNN vs GRU vs LSTM on a long-dependency memory task")
plt.legend()
plt.show()

**Reflections:**
 1) Which model converges fastest? Which reaches the lowest loss?  
 2) How does this relate to vanishing gradients?  
 3) In your own words: what do GRU/LSTM gates help the model do?